# KEGG snapshot 원본 조각: 탐색·저장 셀
추가 첨부의 82번째(In[88])·83번째(In[90]) 셀을 코드와 저장 출력 그대로 보존합니다.
전체 매핑 생성 원본이 아닙니다. columns/audit/pathway_audit/probe_order/PATHWAY_OUT 등의 생성 정의가 없어 독립 실행할 수 없습니다. 셀을 실행하지 마세요. 이번 작업에서는 다운로드·재생성·학습을 수행하지 않았습니다.


In [88]:
from pathlib import Path
import os
import pandas as pd

DATA = Path("/home/seowon1211/data1")

genes = pd.read_csv(DATA / "AIVIVE/data/final_rat_genes.csv")
print("유전자 매핑 컬럼:", genes.columns.tolist())
display(genes.head())

keywords = ("pathway", "kegg", "reactome", "gene2go", "gene_sets")
skip = {
    ".git", ".cache", ".ipynb_checkpoints", "__pycache__",
    "node_modules", "site-packages", "conditioned_expression",
}
found = []

for directory, dirs, files in os.walk(DATA):
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".") and d not in skip
    ]
    for filename in files:
        path = Path(directory) / filename
        name = filename.lower()
        if (
            path.suffix.lower() == ".gmt"
            or (
                any(word in name for word in keywords)
                and path.suffix.lower() in {".csv", ".tsv", ".txt"}
            )
        ):
            found.append(path)

print("\npathway 관련 후보 파일:")
for path in found[:40]:
    print(path)

if not found:
    print("후보 파일 없음 — 외부의 고정된 Rat 유전자 집합이 필요합니다.")

print("\nAIVIVE 공발현 모듈 파일 — pathway와 별도로 취급:")
module_dir = DATA / "AIVIVE/data/modules_genes"
if module_dir.is_dir():
    for path in sorted(module_dir.iterdir())[:5]:
        print(path)

유전자 매핑 컬럼: ['PROBEID', 'GENENAME', 'SYMBOL']


,PROBEID,GENENAME,SYMBOL
0,1367454_at,COPI coat complex subunit beta 2,Copb2
1,1367457_at,beclin 1,Becn1
2,1367459_at,ADP-ribosylation factor 1,Arf1
3,1367464_at,poly-U binding splicing factor 60,Puf60
4,1367468_at,SCAN domain-containing 1,Scand1



pathway 관련 후보 파일:
후보 파일 없음 — 외부의 고정된 Rat 유전자 집합이 필요합니다.

AIVIVE 공발현 모듈 파일 — pathway와 별도로 취급:
/home/seowon1211/data1/AIVIVE/data/modules_genes/module115_genes.csv
/home/seowon1211/data1/AIVIVE/data/modules_genes/module120_genes.csv
/home/seowon1211/data1/AIVIVE/data/modules_genes/module15_genes.csv
/home/seowon1211/data1/AIVIVE/data/modules_genes/module176_genes.csv
/home/seowon1211/data1/AIVIVE/data/modules_genes/module18_genes.csv


In [90]:
# 계산·검증은 float64로 수행
projection = np.column_stack(columns).astype(np.float64)
column_sums = projection.sum(axis=0)

assert np.isfinite(projection).all()
assert (projection >= 0).all()
assert (column_sums > 0).all()

# 정규화 전에 차이가 실제로 반올림 수준인지 확인
np.testing.assert_allclose(
    column_sums, 1.0,
    rtol=0, atol=1e-6,
)

projection = projection / column_sums[None, :]
np.testing.assert_allclose(
    projection.sum(axis=0), 1.0,
    rtol=0, atol=1e-12,
)

audit.to_csv(PATHWAY_OUT / "probe_mapping_audit.csv", index=False)

pathway_table = pd.DataFrame(pathway_audit)
pathway_table.to_csv(PATHWAY_OUT / "pathway_audit.csv", index=False)

np.save(PATHWAY_OUT / "projection.npy", projection)

(PATHWAY_OUT / "provenance.json").write_text(json.dumps({
    "species": "Rattus norvegicus",
    "sources": provenance,
    "minimum_matched_genes": 5,
    "maximum_total_pathway_genes": 500,
    "probe_order": probe_order.tolist(),
    "projection_dtype": "float64",
}, ensure_ascii=False, indent=2))

print("전체 probe:", len(audit))
print("KEGG 유전자에 연결된 probe:", int(audit.kegg_gene.notna().sum()))
print("사용 가능한 pathway:", len(pathway_table))
print("저장 위치:", PATHWAY_OUT)
display(pathway_table.head(10))

전체 probe: 3453
KEGG 유전자에 연결된 probe: 3410
사용 가능한 pathway: 321
저장 위치: /home/seowon1211/data1/conditioned_expression/kegg_rat_20261102_034010_265aed


,pathway,name,total_kegg_genes,matched_genes,matched_probes
0,rno00010,Glycolysis / Gluconeogenesis - Rattus norvegic...,87,25,37
1,rno00020,Citrate cycle (TCA cycle) - Rattus norvegicus ...,32,11,13
2,rno00030,Pentose phosphate pathway - Rattus norvegicus ...,32,14,17
3,rno00040,Pentose and glucuronate interconversions - Rat...,37,8,11
4,rno00051,Fructose and mannose metabolism - Rattus norve...,39,11,18
5,rno00052,Galactose metabolism - Rattus norvegicus (rat),33,12,25
6,rno00053,Ascorbate and aldarate metabolism - Rattus nor...,32,9,11
7,rno00062,Fatty acid elongation - Rattus norvegicus (rat),31,12,16
8,rno00071,Fatty acid degradation - Rattus norvegicus (rat),50,23,32
9,rno00100,Steroid biosynthesis - Rattus norvegicus (rat),20,12,20
